In [1]:
# creating spark session
exec(open('/home/jovyan/.ipython/profile_default/startup/00-spark-session.py').read())

Spark 3.5.0 session ready as `spark` (Delta Lake enabled).


In [2]:
# Adding Display functionality of Databricks 
exec(open('/home/jovyan/.ipython/profile_default/startup/01-databricks-utils.py').read())

Databricks-style helpers ready: display(), dbutils.fs/widgets/notebook, %run_notebook


# Capital Gain Loss

## Problem Description

You are given a PySpark DataFrame named **`stocks`** containing stock trading operations.

The DataFrame has the following columns:

| Column | Data Type | Description |
|---|---|---|
| `stock_name` | string | Name of the stock |
| `operation` | string | Type of operation: `Buy` or `Sell` |
| `operation_day` | int | Day on which the operation occurred |
| `price` | int | Price of the stock during the operation |

### Task

Calculate the **capital gain or loss** for each stock.

The capital gain/loss is calculated as:

```text
Capital Gain/Loss = Total Sell Price - Total Buy Price

In [4]:
data = [
    ("Leetcode", "Buy", 1, 1000),
    ("Leetcode", "Sell", 5, 9000),
    ("Leetcode", "Buy", 6, 4000),
    ("Leetcode", "Sell", 8, 5000),

    ("Corona", "Buy", 2, 3000),
    ("Corona", "Sell", 3, 1200),

    ("Amazon", "Buy", 4, 5000),
    ("Amazon", "Sell", 7, 8000),
    ("Amazon", "Buy", 10, 2000),
    ("Amazon", "Sell", 12, 1500),

    ("Tesla", "Buy", 3, 7000),
    ("Tesla", "Sell", 6, 9000),
    ("Tesla", "Buy", 9, 3000),
    ("Tesla", "Sell", 11, 2500),

    ("Google", "Buy", 2, 4000),
    ("Google", "Sell", 5, 6000),

    ("Microsoft", "Buy", 1, 8000),
    ("Microsoft", "Sell", 4, 7500),
    ("Microsoft", "Buy", 8, 2000),
    ("Microsoft", "Sell", 10, 3500),
]

columns = [
    "stock_name",
    "operation",
    "operation_day",
    "price"
]

stocks = spark.createDataFrame(data, columns)

stocks.show()

+----------+---------+-------------+-----+
|stock_name|operation|operation_day|price|
+----------+---------+-------------+-----+
|  Leetcode|      Buy|            1| 1000|
|  Leetcode|     Sell|            5| 9000|
|  Leetcode|      Buy|            6| 4000|
|  Leetcode|     Sell|            8| 5000|
|    Corona|      Buy|            2| 3000|
|    Corona|     Sell|            3| 1200|
|    Amazon|      Buy|            4| 5000|
|    Amazon|     Sell|            7| 8000|
|    Amazon|      Buy|           10| 2000|
|    Amazon|     Sell|           12| 1500|
|     Tesla|      Buy|            3| 7000|
|     Tesla|     Sell|            6| 9000|
|     Tesla|      Buy|            9| 3000|
|     Tesla|     Sell|           11| 2500|
|    Google|      Buy|            2| 4000|
|    Google|     Sell|            5| 6000|
| Microsoft|      Buy|            1| 8000|
| Microsoft|     Sell|            4| 7500|
| Microsoft|      Buy|            8| 2000|
| Microsoft|     Sell|           10| 3500|
+----------

# Using Spark Sql

In [5]:
stocks.createOrReplaceTempView("stocks")

In [11]:
spark.sql(
    """
        SELECT stock_name,
        sum(
            CASE 
                WHEN operation = 'Buy' THEN -price 
                WHEN operation = 'Sell' THEN price
                ELSE 0
            END
            ) AS capital_gain
        from stocks 
        group by stock_name
    """
    
).show()


+----------+------------+
|stock_name|capital_gain|
+----------+------------+
|    Corona|       -1800|
|  Leetcode|        9000|
|    Amazon|        2500|
|     Tesla|        1500|
|    Google|        2000|
| Microsoft|        1000|
+----------+------------+



# Using Pyspark

In [14]:
from pyspark.sql import functions as F

result = (
    stocks
    .groupBy("stock_name")
    .agg(
        F.sum(
            F.when(F.col("operation") == "Buy", -F.col("price"))
             .when(F.col("operation") == "Sell", F.col("price"))
             .otherwise(0)
        ).alias("capital_gain_loss")
    )
)

result.show()

+----------+-----------------+
|stock_name|capital_gain_loss|
+----------+-----------------+
|    Corona|            -1800|
|  Leetcode|             9000|
|    Amazon|             2500|
|     Tesla|             1500|
|    Google|             2000|
| Microsoft|             1000|
+----------+-----------------+

